# Đọc kết quả A/B chiến dịch quảng cáo: kiểm tra trước, kết luận sau

Một công ty chạy chiến dịch quảng cáo online. Để đo hiệu quả thật, đội marketing giữ lại một nhóm nhỏ người dùng
**không thấy quảng cáo**: ở đúng vị trí đó họ thấy một thông điệp công ích (PSA, public service announcement).
Nhóm `ad` thấy quảng cáo, nhóm `psa` là nhóm đối chứng. Metric chính: `converted` (người dùng có mua hay không).

Giám đốc marketing (CMO) hỏi: **"Chiến dịch có tạo thêm đơn không, bao nhiêu, và có nên tiếp tục?"**

Bộ dữ liệu `marketing-ab` (Kaggle, CC0, 588 101 người dùng). Mỗi dòng một người dùng: `test group`, `converted`,
`total ads` (số quảng cáo hoặc PSA đã thấy), `most ads day`, `most ads hour` (ngày, giờ người đó thấy nhiều
quảng cáo nhất).

Thứ tự làm việc của DA: **kiểm tra độ tin cậy** (SRM, cân bằng giữa hai nhóm) → **ước lượng hiệu ứng có khoảng
tin cậy** → **quyết định theo quy tắc đặt trước** → **ghi rõ giới hạn**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/intermediate/01-marketing-ab-readout -q`

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

COLUMNS = {"user id": "user_id", "test group": "group", "converted": "converted", "total ads": "total_ads",
           "most ads day": "most_ads_day", "most ads hour": "most_ads_hour"}


def read_ab(path) -> pd.DataFrame:
    """Đọc marketing_AB.csv, bỏ cột chỉ số thừa, đổi tên cột sang snake_case. (Đã viết sẵn.)"""
    return pd.read_csv(path).rename(columns=COLUMNS)[list(COLUMNS.values())]

## Phần A. Tóm tắt và kiểm tra SRM

`summarize(df) -> DataFrame`: mỗi nhóm (`group`) một dòng, index tên nhóm sắp theo chữ cái, cột `users` (số người),
`conversions` (số `converted` đúng), `rate` (`conversions / users`).

`srm_check(observed, expected_share) -> dict` với khoá `chi2`, `p`: kiểm định chi-square goodness of fit giữa số
người quan sát mỗi nhóm (`observed`: Series hoặc dict `nhóm -> số người`) và số người **kỳ vọng** theo tỉ lệ chia
đã thiết kế (`expected_share`: dict `nhóm -> tỉ lệ`, cộng lại bằng 1). Sample Ratio Mismatch (SRM) khi p rất nhỏ
(thường dùng ngưỡng 0.001): cách chia người dùng hoặc cách ghi log có vấn đề, mọi kết quả phía sau không tin được.

`segment_balance(df, segment) -> dict` với khoá `chi2`, `p`, `dof`: chi-square độc lập trên bảng chéo
`segment × group` (`scipy.stats.chi2_contingency`, `correction=False`). Nếu chia ngẫu nhiên đúng và hai nhóm được
phục vụ như nhau, tỉ lệ `psa` phải gần như nhau ở mọi giá trị của `segment`.

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Mỗi nhóm: users, conversions, rate."""
    out = df.groupby("group").agg(users=("converted", "size"), conversions=("converted", "sum"))
    out["conversions"] = out["conversions"].astype(int)
    out["rate"] = out["conversions"] / out["users"]
    return out.sort_index()


def srm_check(observed, expected_share: dict) -> dict:
    """Chi-square goodness of fit giữa số người mỗi nhóm và tỉ lệ thiết kế: chi2, p."""
    observed = pd.Series(observed, dtype=float)
    groups = list(expected_share)
    obs = observed.loc[groups].to_numpy()
    expected = np.array([expected_share[g] for g in groups]) * obs.sum()
    res = stats.chisquare(obs, expected)
    return {"chi2": float(res.statistic), "p": float(res.pvalue)}


def segment_balance(df: pd.DataFrame, segment: str) -> dict:
    """Chi-square độc lập trên bảng chéo segment × group: chi2, p, dof."""
    table = pd.crosstab(df[segment], df["group"])
    chi2, p, dof, _ = stats.chi2_contingency(table, correction=False)
    return {"chi2": float(chi2), "p": float(p), "dof": int(dof)}

In [ ]:
# explore: đọc dữ liệu thật, tóm tắt, SRM theo tỉ lệ thiết kế 96/4
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

ab = read_ab(ensure("marketing-ab") / "marketing_AB.csv")
print(f"{len(ab):,} người dùng, {ab.user_id.nunique():,} user_id khác nhau")
summary = summarize(ab)
print(summary.assign(share=summary.users / summary.users.sum()).round(5).to_string())

DESIGN = {"ad": 0.96, "psa": 0.04}  # tỉ lệ chia đội marketing ghi trong brief
print("SRM so với 96/4:", srm_check(summary["users"], DESIGN))
print("Nếu brief ghi 95/5:", srm_check(summary["users"], {"ad": 0.95, "psa": 0.05}))

In [ ]:
# explore: cân bằng giữa hai nhóm theo ngày và giờ người dùng thấy nhiều quảng cáo nhất
for seg in ["most_ads_day", "most_ads_hour"]:
    print(seg, segment_balance(ab, seg))
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
psa_share = pd.crosstab(ab.most_ads_day, ab.group, normalize="index")["psa"].reindex(order)
print((psa_share * 100).round(2).rename("% psa").to_string())

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(psa_share.index, psa_share.values * 100, color="#4c78a8")
ax.axhline(4, color="#e45756", lw=1, ls="--")
ax.annotate("thiết kế 4%", (6.1, 4.05), color="#e45756", ha="right", va="bottom")
ax.set_ylim(0, 5.5)
ax.set_ylabel("% người dùng thuộc nhóm psa")
ax.set_xlabel("ngày thấy nhiều quảng cáo nhất")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Tổng thể đúng 96/4, nhưng tỉ lệ psa theo ngày dao động 3.5-4.7%", loc="left")
plt.show()

## Phần B. Hiệu ứng, khoảng tin cậy và quyết định

`diff_ci(conv_t, n_t, conv_c, n_c, alpha=0.05) -> dict` với khoá `rate_t`, `rate_c`, `diff` (`rate_t - rate_c`),
`ci_low`, `ci_high`, `rel_lift`, `p`:

- Khoảng tin cậy của `diff` theo Wald, phương sai **riêng** từng nhóm:
  $$\text{diff} \pm z_{1-\alpha/2}\sqrt{\frac{p_t(1-p_t)}{n_t} + \frac{p_c(1-p_c)}{n_c}}$$
- `p`: z-test hai phía với tỉ lệ **gộp** $$\hat p = (x_t + x_c)/(n_t + n_c)$$ dưới H0, sai số chuẩn
  $$\sqrt{\hat p(1-\hat p)(1/n_t + 1/n_c)}$$ (giống `statsmodels.stats.proportion.proportions_ztest`).
- `rel_lift = diff / rate_c`.

`decide(ci_low, ci_high, mde, srm_ok, guardrail_ok=True) -> str`, quy tắc **viết trước khi xem kết quả**, xét theo
thứ tự:

1. `srm_ok` sai → `"investigate"` (dữ liệu chưa tin được, chưa kết luận gì).
2. `guardrail_ok` sai → `"do_not_ship"` (metric bảo vệ bị hại).
3. `ci_high < mde` → `"do_not_ship"` (kể cả khi có ý nghĩa thống kê, hiệu ứng chắc chắn nhỏ hơn mức đáng làm).
4. `ci_low > 0` → `"ship"`.
5. Còn lại → `"inconclusive"` (khoảng tin cậy chứa 0 nhưng vẫn có thể đạt `mde`: thiếu dữ liệu).

In [ ]:
def diff_ci(conv_t: int, n_t: int, conv_c: int, n_c: int, alpha: float = 0.05) -> dict:
    """Chênh lệch tỉ lệ (t - c), CI Wald phương sai riêng, p z-test tỉ lệ gộp, lift tương đối."""
    p_t, p_c = conv_t / n_t, conv_c / n_c
    diff = p_t - p_c
    se = math.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
    crit = stats.norm.ppf(1 - alpha / 2)
    pooled = (conv_t + conv_c) / (n_t + n_c)
    se0 = math.sqrt(pooled * (1 - pooled) * (1 / n_t + 1 / n_c))
    p = float(2 * stats.norm.sf(abs(diff / se0)))
    return {"rate_t": p_t, "rate_c": p_c, "diff": diff, "ci_low": diff - crit * se, "ci_high": diff + crit * se,
            "rel_lift": diff / p_c, "p": p}


def decide(ci_low: float, ci_high: float, mde: float, srm_ok: bool, guardrail_ok: bool = True) -> str:
    """'investigate' | 'do_not_ship' | 'ship' | 'inconclusive' theo quy tắc trong đề."""
    if not srm_ok:
        return "investigate"
    if not guardrail_ok:
        return "do_not_ship"
    if ci_high < mde:
        return "do_not_ship"
    if ci_low > 0:
        return "ship"
    return "inconclusive"

In [ ]:
# explore: hiệu ứng tổng, quyết định, và hiệu ứng theo ngày
t, c = summary.loc["ad"], summary.loc["psa"]
res = diff_ci(t.conversions, t.users, c.conversions, c.users)
print({k: round(v, 6) for k, v in res.items()})
print(f"chênh lệch {res['diff'] * 100:.3f} điểm % [{res['ci_low'] * 100:.3f}; {res['ci_high'] * 100:.3f}], "
      f"tương đối {res['rel_lift']:.1%}")
print(f"đơn tăng thêm ước tính trong nhóm ad: {res['diff'] * t.users:,.0f} "
      f"[{res['ci_low'] * t.users:,.0f}; {res['ci_high'] * t.users:,.0f}]")

MDE = 0.005  # ví dụ giả định: CMO cho rằng chiến dịch đáng tiền khi tăng ít nhất 0.5 điểm %
srm_ok = srm_check(summary["users"], DESIGN)["p"] >= 0.001
print("quyết định:", decide(res["ci_low"], res["ci_high"], MDE, srm_ok))

by_day = pd.DataFrame({
    d: diff_ci(g[g.group == "ad"].converted.sum(), (g.group == "ad").sum(),
               g[g.group == "psa"].converted.sum(), (g.group == "psa").sum())
    for d, g in ab.groupby("most_ads_day")
}).T.reindex(order)
print((by_day[["rate_t", "rate_c", "diff", "ci_low", "ci_high", "p"]] * [100, 100, 100, 100, 100, 1]).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.5))
y = np.arange(len(by_day))
ax.errorbar(by_day["diff"] * 100, y, xerr=[(by_day["diff"] - by_day["ci_low"]) * 100,
            (by_day["ci_high"] - by_day["diff"]) * 100], fmt="o", color="#4c78a8", capsize=3)
ax.axvline(0, color="#333", lw=0.8)
ax.axvline(res["diff"] * 100, color="#e45756", lw=1, ls="--")
ax.set_yticks(y, by_day.index)
ax.invert_yaxis()
ax.set_xlabel("chênh lệch tỉ lệ chuyển đổi ad - psa (điểm %)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Hiệu ứng dương ở cả 7 ngày; khoảng tin cậy từng ngày rộng vì nhóm psa nhỏ", loc="left")
plt.show()

In [ ]:
# explore: người thấy nhiều quảng cáo chuyển đổi nhiều hơn, ở CẢ HAI nhóm
bins = [0, 5, 10, 20, 50, 100, np.inf]
labels = ["1-5", "6-10", "11-20", "21-50", "51-100", ">100"]
dose = (ab.assign(bucket=pd.cut(ab.total_ads, bins, labels=labels))
          .groupby(["bucket", "group"], observed=True).converted.agg(["mean", "size"]).unstack())
print((dose["mean"] * 100).round(2).to_string())
print(dose["size"].to_string())
print("tương quan total_ads - converted trong nhóm psa:",
      round(ab[ab.group == "psa"][["total_ads", "converted"]].astype(float).corr().iloc[0, 1], 3))

## Đọc kết quả, rồi viết ba câu cho CMO

1. SRM tổng thể so với thiết kế 96/4 cho p bao nhiêu? Nếu brief ghi nhầm 95/5 thì sao? Vì sao DA phải lấy tỉ lệ
   thiết kế từ brief chứ không từ dữ liệu?
2. Tỉ lệ `psa` theo ngày lệch khỏi 4% thế nào, p bao nhiêu? `most_ads_day` đo **sau** khi chia nhóm (ngày người
   dùng thấy nhiều quảng cáo nhất): điều đó nói gì về nguyên nhân có thể (chia nhóm hỏng, hay quảng cáo và PSA
   được phân phối khác nhau theo ngày)? Ai cần được hỏi?
3. Hiệu ứng tổng: bao nhiêu điểm %, tương đối bao nhiêu, khoảng tin cậy? Quyết định theo quy tắc là gì?
4. Bảng theo `total_ads`: người thấy hơn 100 quảng cáo chuyển đổi 17%, người thấy 1-5 chỉ 0.25%. Có được nói
   "mỗi quảng cáo thêm làm tăng chuyển đổi" không? Nhìn nhóm `psa` để trả lời.

Ba câu cho CMO: có tạo thêm đơn không (con số và khoảng), có nên tiếp tục không, và một điều cần kiểm thêm.

Xong thì mở app, làm phần **trắc nghiệm** của bài.